# 10월 12일 과제 — RAG 기초부터 의미 검색까지 한 번에

**한빛시립도서관 이용 안내 챗봇 '책봇' 만들기**

> ⚠️ 도서관 이름과 규정은 모두 **교육용으로 만든 가상 내용**입니다.

## 시나리오

한빛시립도서관 안내데스크에는 같은 질문이 하루 수백 번 들어옵니다.
"책 늦게 내면 벌금 있어요?", "스터디룸 오늘 예약돼요?", "주차 몇 시간 무료예요?"

사서들의 부담을 줄이기 위해, 도서관 규정 문서를 보고 답하는 **'책봇'** 을 만듭니다.
오늘 배운 3개 차시(현경봇)를 **새 시나리오로 처음부터 다시** 만들어 보는 복습입니다.

| 파트 | 복습 내용 | 대응 차시 |
|------|-----------|-----------|
| A | 환각 확인 → 프롬프트 → 문서 통째로 넣기(RAG) | Day5_1 |
| B | 파일 불러오기 → 쪼개기 → 꼬리표 | Day5_2 |
| C | 임베딩 → 벡터 DB → 의미 검색·필터·점수·MMR | Day5_3 |
| D | 🏁 종합: 검색한 청크로 **출처 붙은 답변** 만들기 | 다음 차시 미리보기 |

## 도서관 문서 5개

| 파일 | 내용 | 담당 |
|------|------|------|
| `01_faq.md` | 운영시간, 분실, 와이파이, 희망도서 | 안내데스크 |
| `02_loan.md` | 대출 권수·기간 표, 연장, 반납, 연체, 예약, 상호대차 | 자료봉사팀 |
| `03_facility.md` | 열람실, 스터디룸 표, 노트북 존, 주차 | 시설운영팀 |
| `04_membership.md` | 가입 대상, 회원증, 가족 회원, 자격 유지 | 자료봉사팀 |
| `05_program.md` | 정기 프로그램 표, 신청·취소, 독서 마라톤 | 문화프로그램팀 |

## 📝 진행 방법

- `____` (밑줄 4개)를 알맞은 코드로 바꾼 뒤 실행하세요. 문제마다 💡 힌트가 있습니다.
- 막히면 오늘 만든 **현경봇 노트북**의 같은 단계를 참고하세요.
- 노트북 **맨 아래에 정답**이 있습니다. 먼저 스스로 풀어 보세요!
- 파트 A와 D는 **OpenAI 키가 필요**합니다. 파트 B, C는 키 없이도 됩니다(무료 임베딩).

# 🔧 준비

### 준비 1 — 도구 설치

In [ ]:
!pip install -q langchain langchain-openai langchain-community langchain-chroma langchain-text-splitters langchain-huggingface python-dotenv sentence-transformers

### 준비 2 — 키, AI 모델, 임베딩 모델 준비

In [1]:
import os

try:
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
except Exception:
    from dotenv import load_dotenv
    load_dotenv()

# LangSmith 키가 없을 때 401 경고 방지
os.environ["LANGCHAIN_TRACING_V2"] = "false"
os.environ["LANGSMITH_TRACING"] = "false"

if os.environ.get("OPENAI_API_KEY"):
    from langchain_openai import ChatOpenAI, OpenAIEmbeddings
    llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
    embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
    print("✅ OpenAI 모델 준비 완료 (파트 A~D 모두 가능)")
else:
    from langchain_huggingface import HuggingFaceEmbeddings
    llm = None
    embeddings = HuggingFaceEmbeddings(
        model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2",
        encode_kwargs={"normalize_embeddings": True},
    )
    print("⚠️ 키 없음: 무료 임베딩만 준비 (파트 B, C만 가능)")

✅ OpenAI 모델 준비 완료 (파트 A~D 모두 가능)


### 준비 3 — 도서관 문서 5개를 변수에 담기

파트 A는 이 변수를 그대로 쓰고, 파트 B에서는 파일로 저장해 불러옵니다.

In [2]:
import os

# 한빛시립도서관 문서 5개 (※ 교육용 가상 문서)
documents = {
    "01_faq.md": """# 한빛시립도서관 자주 묻는 질문 (FAQ)

※ 교육용 가상 문서입니다. 실제 도서관과 무관합니다.

### Q. 도서관 운영시간은 어떻게 되나요?
A. 자료실은 화~금 09:00~20:00, 토·일 09:00~17:00 운영합니다.
매주 **월요일과 법정 공휴일은 휴관**입니다.

### Q. 회원증 없이 책을 빌릴 수 있나요?
A. 아니요. 대출하려면 회원증 또는 모바일 회원증이 필요합니다.
회원증은 1층 안내데스크에서 신분증을 보여주시면 바로 발급합니다.

### Q. 책을 잃어버렸어요.
A. 같은 책(동일 판본)으로 변상하거나, 구입이 어려우면 정가의 1.5배를 현금으로 변상합니다.
변상 전까지는 대출이 정지됩니다.

### Q. 와이파이를 쓸 수 있나요?
A. 네. 'HANBIT_LIB'에 연결한 뒤 회원번호로 로그인하면 됩니다. 1회 접속 시 3시간까지 사용합니다.

### Q. 음식물을 가지고 들어가도 되나요?
A. 자료실과 열람실에는 뚜껑 있는 음료만 반입할 수 있습니다. 음식은 2층 휴게실에서만 드실 수 있습니다.

### Q. 희망도서를 신청하고 싶어요.
A. 홈페이지 > 희망도서 신청에서 1인 월 3권까지 신청할 수 있습니다.
입고되면 문자로 알려드리며, 신청자에게 3일간 우선 대출 기회가 주어집니다.

### Q. 문의 전화번호를 알려주세요.
A. 대표번호 031-000-1000, 자료실 031-000-1100, 문화프로그램 031-000-1200입니다.
""",

    "02_loan.md": """# 한빛시립도서관 대출·반납 규정

※ 교육용 가상 문서입니다. 실제 도서관과 무관합니다.

담당: 자료봉사팀

## 제1조 대출 권수와 기간

| 회원 구분 | 대출 권수 | 대출 기간 |
|-----------|-----------|-----------|
| 일반 회원 | 1인 7권 | 14일 |
| 어린이 회원(13세 미만) | 1인 5권 | 14일 |
| 가족 회원 | 가족 합산 최대 20권 | 14일 |
| 장애인·국가유공자 | 1인 10권 | 28일 |

- DVD 등 영상자료는 1인 2점, 7일간 대출합니다.
- 신착 도서(입고 1개월 이내)는 1인 2권까지만 대출할 수 있습니다.

## 제2조 대출 연장

- 반납 예정일 전에 **1회, 7일** 연장할 수 있습니다.
- 다른 회원이 예약한 자료는 연장할 수 없습니다.
- 연체 중인 자료가 있으면 연장할 수 없습니다.
- 연장은 홈페이지, 모바일 앱, 안내데스크에서 할 수 있습니다.

## 제3조 반납

- 운영시간에는 자료실 반납대에, 운영시간 외에는 정문 옆 **무인반납함**에 반납합니다.
- DVD 등 영상자료는 파손 위험이 있어 무인반납함에 넣을 수 없습니다.
- **타관 반납**: 시내 다른 공공도서관 5곳에서도 반납할 수 있으며, 처리까지 2~3일이 걸립니다.

## 제4조 연체

- 반납 예정일을 넘기면 **연체 일수만큼 대출이 정지**됩니다.
  예) 5일 연체 → 반납한 날부터 5일간 대출 불가
- 연체 정지 기간은 최대 60일입니다.
- 연체료(돈)는 받지 않습니다.
- 휴관일은 연체 일수에 포함되지 않습니다.

## 제5조 예약

- 대출 중인 자료는 1인 3권까지 예약할 수 있습니다.
- 예약 자료가 반납되면 문자로 알리며, **3일 안에** 찾아가지 않으면 예약이 자동 취소됩니다.
- 예약 후 찾아가지 않는 일이 3회 쌓이면 1개월간 예약할 수 없습니다.

## 제6조 상호대차

- 우리 도서관에 없는 책은 시내 다른 공공도서관에서 빌려 올 수 있습니다(상호대차).
- 1인 월 5권까지 신청할 수 있으며, 도착까지 3~5일이 걸립니다.
- 상호대차 자료는 연장할 수 없습니다.
""",

    "03_facility.md": """# 한빛시립도서관 시설 이용 규정

※ 교육용 가상 문서입니다. 실제 도서관과 무관합니다.

담당: 시설운영팀

## 제1조 열람실

- 운영시간: 화~일 07:00~23:00 (월요일 휴실)
- 좌석은 키오스크나 모바일 앱으로 배정받으며, 1회 4시간 이용 후 연장할 수 있습니다.
- 자리를 1시간 이상 비우면 좌석이 자동으로 반납됩니다.
- 시험 기간(중간·기말고사 2주 전)에는 1인 1일 8시간까지로 제한합니다.

## 제2조 스터디룸

| 구분 | 인원 | 이용 시간 |
|------|------|-----------|
| 소형 스터디룸 | 2~4인 | 1회 2시간 |
| 대형 스터디룸 | 5~8인 | 1회 3시간 |

- 홈페이지에서 **7일 전부터 전날까지** 예약할 수 있습니다. 당일 예약은 빈 방이 있을 때 안내데스크에서만 가능합니다.
- 1인 주 2회까지 예약할 수 있습니다.
- 예약 시간 15분이 지나도 오지 않으면 예약이 취소되고, 2회 이상이면 1개월간 예약할 수 없습니다.

## 제3조 노트북 존·멀티미디어실

- 노트북 존(3층)은 전원 콘센트가 있는 좌석 40석이며, 자유석입니다.
- 멀티미디어실 PC는 1회 2시간, 1일 2회까지 사용할 수 있습니다.
- 출력은 흑백 1장 100원, 컬러 1장 300원이며 교통카드로 결제합니다.

## 제4조 주차

- 주차장은 60면이며, 도서관 이용자는 **2시간 무료**입니다.
- 2시간 이후 30분마다 500원이 부과됩니다.
- 무료 주차를 받으려면 출차 전 안내데스크에서 회원증을 확인받아야 합니다.
- 장애인·임산부·다자녀 가정은 4시간 무료입니다.

## 제5조 이용 제한

- 열람실·자료실에서 통화, 큰 소리 대화, 좌석 맡아두기는 금지입니다.
- 위반 시 1차 경고, 2차 7일 이용 정지, 3차 1개월 이용 정지합니다.
""",

    "04_membership.md": """# 한빛시립도서관 회원 규정

※ 교육용 가상 문서입니다. 실제 도서관과 무관합니다.

담당: 자료봉사팀

## 제1조 가입 대상

- 한빛시에 거주하거나, 한빛시 소재 직장·학교에 다니는 사람은 누구나 가입할 수 있습니다.
- 다른 지역 거주자는 '준회원'으로 가입하며, 대출은 1인 3권까지 가능합니다.

## 제2조 회원증 발급

- 1층 안내데스크에서 신분증을 확인한 뒤 바로 발급합니다. 발급비는 무료입니다.
- 14세 미만 어린이는 보호자가 함께 와서 가족관계를 확인해야 합니다.
- 모바일 회원증은 앱에서 본인인증 후 바로 쓸 수 있습니다.
- 분실 재발급은 1회 무료, 2회부터 1,000원입니다.

## 제3조 가족 회원

- 주민등록상 같은 세대의 가족은 가족 회원으로 묶을 수 있습니다(최대 6명).
- 가족 회원은 대출 권수를 합산해 최대 20권까지 빌릴 수 있습니다.
- 가족 중 1명이 연체하면 가족 전체가 아닌 **연체한 본인만** 대출이 정지됩니다.

## 제4조 회원 자격 유지

- 마지막 이용일로부터 **3년** 동안 이용 기록이 없으면 회원 자격이 자동 해지되고, 개인정보는 파기됩니다.
- 해지 30일 전 문자로 알려드립니다.

## 제5조 개인정보

- 대출 기록은 반납 후 즉시 삭제하는 것이 원칙입니다.
- 본인이 동의하면 '나의 독서 기록'으로 대출 이력을 보관할 수 있습니다.
""",

    "05_program.md": """# 한빛시립도서관 문화프로그램 안내

※ 교육용 가상 문서입니다. 실제 도서관과 무관합니다.

담당: 문화프로그램팀

## 1. 정기 프로그램

| 프로그램 | 대상 | 일정 | 정원 |
|----------|------|------|------|
| 그림책 읽어주기 | 4~7세 | 매주 토 10:30 | 15명 |
| 어린이 독서교실 | 초등 3~6학년 | 방학 중 2주 과정 | 20명 |
| 인문학 강좌 | 성인 | 매월 둘째 주 수 19:00 | 40명 |
| 디지털 배움터 | 60세 이상 | 매주 화·목 14:00 | 12명 |

- 모든 정기 프로그램은 **무료**입니다. (재료비가 있는 경우 별도 안내)

## 2. 신청 방법

- 홈페이지 > 문화행사 > 프로그램 신청에서 선착순으로 접수합니다.
- 접수는 프로그램 시작 **2주 전 오전 10시**에 열립니다.
- 정원이 차면 대기자로 등록되며, 취소자가 생기면 순서대로 연락합니다.
- 1인 같은 달에 최대 2개 프로그램까지 신청할 수 있습니다.

## 3. 취소와 불참

- 프로그램 시작 전날 18시까지 홈페이지에서 취소할 수 있습니다.
- 연락 없이 2회 불참하면 3개월간 프로그램을 신청할 수 없습니다.

## 4. 독서 마라톤

- 매년 3월~11월 운영하며, 읽은 책 쪽수를 앱에 기록합니다.
- 10,000쪽 달성 시 완주 인증서와 도서상품권 1만원을 드립니다.
""",

}

for name, text in documents.items():
    print(f"{name:<18} {len(text):>5}자")

01_faq.md            726자
02_loan.md          1059자
03_facility.md       907자
04_membership.md     685자
05_program.md        699자


---
# 🅰️ 파트 A — 환각 확인과 문서 통째로 넣기 (Day5_1 복습)

### 문제 A-1 — AI는 우리 도서관 규정을 모른다

규정에는 **"연체료(돈)는 받지 않고, 연체 일수만큼 대출 정지"** 라고 적혀 있습니다. AI는 어떻게 답할까요?

In [3]:
question = "한빛시립도서관에서 책을 5일 연체하면 연체료가 얼마예요?"

# TODO: llm에게 질문하는 메서드
answer = llm.invoke(question)

# TODO: 답 글자가 들어 있는 속성
print(answer.content)

한빛시립도서관의 연체료는 도서관의 정책에 따라 다를 수 있지만, 일반적으로 연체료는 하루에 일정 금액이 부과됩니다. 예를 들어, 하루에 100원이 부과된다면 5일 연체 시 500원이 될 것입니다. 정확한 연체료는 한빛시립도서관의 공식 웹사이트나 도서관에 직접 문의하여 확인하는 것이 가장 좋습니다.


> ✍️ AI가 지어낸 부분은? (정답: 연체료 없음, 반납일부터 5일간 대출 정지)

### 문제 A-2 — '책봇' 역할 주기 (프롬프트)

💡 힌트: `ChatPromptTemplate.from_messages`, `system`/`human`, `prompt | llm | StrOutputParser()`

In [4]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

prompt = ChatPromptTemplate.from_messages([
    ("system", "당신은 한빛사립도서관 안내 챗봇입니다. 친절하고 정확한 정보를 제공해 주세요."),        # TODO: 책봇의 역할과 말투 (예: 한빛시립도서관 안내 챗봇, 친절한 존댓말)
    ("human", "{question}"),       # TODO: 질문이 들어갈 자리 이름
])

# TODO: 프롬프트 → AI → 글자만 추출
chatbot = prompt | llm | StrOutputParser()

print(chatbot.invoke({"question": "DVD도 무인반납함에 넣어도 되나요?"}))

네, 한빛사립도서관의 무인반납함에 DVD를 반납하실 수 있습니다. DVD를 반납하실 때는 케이스에 잘 넣어주시고, 손상이 가지 않도록 주의해 주세요. 추가적인 질문이 있으시면 언제든지 말씀해 주세요!


### 문제 A-3 — 규정 문서를 같이 주기 (RAG)

프롬프트에 문서 자리 `{document}`를 만들고, **규정에 없으면 모른다고** 하게 합니다.

In [7]:
rag_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "당신은 한빛시립도서관 안내 챗봇 '책봇'입니다. "
     "[규정]만 보고 답하고, 없으면 '규정에서 확인되지 않습니다'라고 답해주세요.\n\n"
     "[규정]\n{document}"),
    ("human", "{question}"),
])
rag_chatbot = rag_prompt | llm | StrOutputParser()

question = "한빛시립도서관에서 책을 5일 연체하면 연체료가 얼마예요?"

answer = rag_chatbot.invoke({
    "document": documents["02_loan.md"],     # TODO: 연체 내용이 있는 파일 이름
    "question": question,
})
print(answer)

연체료(돈)는 받지 않습니다.


### 문제 A-4 — 문서 없이 vs 있이 비교

질문: **"DVD도 무인반납함에 넣어도 되나요?"** (정답: 안 됨, 파손 위험)

In [8]:
q = "DVD도 무인반납함에 넣어도 되나요?"

print("───[ 문서 없이 ]───")
print(chatbot.invoke({"question": q}))                                  # TODO: A-2의 챗봇

print("\n───[ 문서 있이 (RAG) ]───")
print(rag_chatbot.invoke({"document": documents["02_loan.md"], "question": q}))   # TODO: A-3의 챗봇

───[ 문서 없이 ]───
네, DVD는 무인반납함에 넣으셔도 됩니다. 반납 시 DVD가 손상되지 않도록 주의해 주세요. 추가로 궁금한 점이 있으시면 언제든지 문의해 주세요!

───[ 문서 있이 (RAG) ]───
DVD 등 영상자료는 파손 위험이 있어 무인반납함에 넣을 수 없습니다.


### 문제 A-5 — 통째로 넣기의 비용

문서 5개를 모두 넣을 때와 1개만 넣을 때의 입력 토큰을 비교합니다.

In [9]:
all_text = "\n\n".join(documents.values())       # TODO: 딕셔너리의 '내용'만 꺼내는 메서드
q = "스터디룸은 며칠 전부터 예약할 수 있어요?"

one = (rag_prompt | llm).invoke({"document": documents["03_facility.md"], "question": q})
alldoc = (rag_prompt | llm).invoke({"document": all_text, "question": q})

print("문서 1개:", one.usage_metadata["input_tokens"], "토큰")       # TODO: 입력 토큰 키 이름
print("문서 5개:", alldoc.usage_metadata["input_tokens"], "토큰")
print("\n답변:", one.content)

문서 1개: 625 토큰
문서 5개: 2423 토큰

답변: 스터디룸은 홈페이지에서 7일 전부터 전날까지 예약할 수 있습니다.


> 💡 관계없는 문서까지 넣으면 토큰이 몇 배로 늘어납니다. → 파트 B에서 쪼개고, 파트 C에서 필요한 조각만 찾습니다.

---
# 🅱️ 파트 B — 불러오기·쪼개기·꼬리표 (Day5_2 복습)

### 문제 B-1 — 문서를 파일로 저장하고 폴더째 불러오기

In [10]:
from langchain_community.document_loaders import DirectoryLoader, TextLoader

os.makedirs("lib_docs", exist_ok=True)
for filename, content in documents.items():
    with open(f"lib_docs/{filename}", "w", encoding="utf-8") as f:
        f.write(content)

loader = DirectoryLoader(
    "lib_docs",                    # TODO: 폴더 이름
    glob="*.md",               # TODO: .md 파일 전부
    loader_cls=TextLoader,           # TODO: 파일 하나를 읽는 로더
    loader_kwargs={"encoding": "utf-8"},
)
all_docs = loader.load()       # TODO: 불러오기

print("불러온 문서 수:", len(all_docs))
print("첫 문서 꼬리표:", all_docs[0].metadata)

불러온 문서 수: 5
첫 문서 꼬리표: {'source': 'lib_docs\\01_faq.md'}


C:\Users\user\AppData\Local\Temp\ipykernel_12740\4242758380.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import DirectoryLoader, TextLoader


### 문제 B-2 — chunk_size별 청크 수와 '표 지키기'

대출 규정의 **대출 권수 표**(머리글 `회원 구분`과 `가족 회원` 행)가 같은 청크에 남는지 확인합니다.

In [11]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

for size in [100, 200, 300, 500]:
    sp = RecursiveCharacterTextSplitter(chunk_size=size, chunk_overlap=30)   # TODO
    cs = sp.split_documents(all_docs)                                                     # TODO: Document 리스트 쪼개기
    together = any("회원 구분" in c.page_content and "가족 회원" in c.page_content for c in cs)
    print(f"chunk_size={size:>3} → 청크 {len(cs):>3}개 | 표 머리글+가족회원 행 함께: {'✅' if together else '❌'}")

chunk_size=100 → 청크  66개 | 표 머리글+가족회원 행 함께: ❌
chunk_size=200 → 청크  29개 | 표 머리글+가족회원 행 함께: ✅
chunk_size=300 → 청크  20개 | 표 머리글+가족회원 행 함께: ✅
chunk_size=500 → 청크  12개 | 표 머리글+가족회원 행 함께: ✅


> ✍️ 표를 지키면서 너무 크지 않은 크기는? 아래 문제부터는 **300**을 씁니다.

### 문제 B-3 — 300자로 쪼개고 꼬리표 2개 달기 (분류 + 담당팀)

In [12]:
splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=30)
chunks = splitter.split_documents(all_docs)

category_map = {
    "01_faq.md": "faq",
    "02_loan.md": "loan",
    "03_facility.md": "facility",          # TODO
    "04_membership.md": "membership",        # TODO
    "05_program.md": "program",
}
team_map = {
    "faq": "안내데스크",
    "loan": "자료봉사팀",
    "facility": "시설담당팀",               # TODO: 시설 담당 팀 (03_facility.md 맨 위 참고)
    "membership": "자료봉사팀",
    "program": "프로그램팀",                # TODO
}

for c in chunks:
    filename = os.path.basename(c.metadata["source"])     # TODO: 파일 경로가 든 꼬리표 키
    c.metadata["category"] = category_map[filename]
    c.metadata["team"] = team_map[c.metadata["category"]]

from collections import Counter
print("전체 청크:", len(chunks))
print("분류별:", dict(Counter(c.metadata["category"] for c in chunks)))
print("예시 꼬리표:", chunks[0].metadata)

전체 청크: 20
분류별: {'faq': 3, 'loan': 5, 'facility': 4, 'membership': 4, 'program': 4}
예시 꼬리표: {'source': 'lib_docs\\01_faq.md', 'category': 'faq', 'team': '안내데스크'}


### 문제 B-4 — 키워드로 찾기와 그 한계

같은 내용을 두 가지 말투로 찾아봅니다. 규정에는 '연체'라고 적혀 있습니다.

In [13]:
def keyword_find(keyword):
    # TODO: 본문에 keyword가 들어간 청크만
    found = [c for c in chunks if keyword in c.page_content]
    print(f"🔎 '{keyword}' → {len(found)}개", [c.metadata["category"] for c in found])

keyword_find("연체")         # 규정집 말투
keyword_find("늦게")         # 이용자 말투: "책 늦게 돌려주면?"

🔎 '연체' → 4개 ['loan', 'loan', 'loan', 'membership']
🔎 '늦게' → 0개 []


> 💡 '늦게'로는 연체 조항을 못 찾습니다. 이용자는 규정집 말투로 묻지 않으니까요. → 파트 C의 **의미 검색**으로 해결합니다.

---
# 🅲 파트 C — 임베딩과 벡터 DB (Day5_3 복습)

### 문제 C-1 — 뜻이 가까우면 점수가 높다 (코사인 유사도)

In [14]:
import numpy as np

def similarity(t1, t2):
    v1 = np.array(embeddings.embed_query(t1))      # TODO: 문장 하나를 임베딩하는 메서드
    v2 = np.array(embeddings.embed_query(t2))
    return float(v1 @ v2 / (np.linalg.norm(v1) * np.linalg.norm(v2)))

base = "책 늦게 돌려주면 어떻게 돼요?"
for other in ["연체 시 대출 정지", "스터디룸 예약 방법", "주차 요금 안내"]:
    print(f"'{base}' ↔ '{other}': {similarity(base, other):.3f}")

'책 늦게 돌려주면 어떻게 돼요?' ↔ '연체 시 대출 정지': 0.131
'책 늦게 돌려주면 어떻게 돼요?' ↔ '스터디룸 예약 방법': 0.128
'책 늦게 돌려주면 어떻게 돼요?' ↔ '주차 요금 안내': 0.209


### 문제 C-2 — 청크를 벡터 DB(Chroma)에 저장

💡 `ids`를 주면 다시 실행해도 중복 저장되지 않고, `hnsw:space`를 cosine으로 하면 점수가 0~1로 나옵니다.

In [15]:
from langchain_chroma import Chroma

lib_db = Chroma.from_documents(                                   # TODO: Document 리스트로 DB 만들기
    chunks, embeddings,                                    # TODO: 파트 B의 청크
    collection_name="hanbit_library",
    collection_metadata={"hnsw:space": "cosine"},          # TODO: 코사인
    ids=[f"chunk-{i}" for i in range(len(chunks))],
)
print("저장된 청크 수:", lib_db._collection.count())

저장된 청크 수: 20


### 문제 C-3 — 의미 검색: B-4에서 실패한 질문 다시 찾기

In [16]:
query = "책 늦게 돌려주면 벌금 있어요?"

results = lib_db.similarity_search(query, k=3)        # TODO: 의미 검색 메서드

print("질문:", query, "\n")
for i, r in enumerate(results, 1):
    print(f"[{i}] ({r.metadata['category']} / 문의: {r.metadata['source']})")   # TODO: 담당팀 꼬리표
    print("   ", r.page_content[:70].replace("\n", " "), "...")

질문: 책 늦게 돌려주면 벌금 있어요? 

[1] (program / 문의: lib_docs\05_program.md)
    ## 4. 독서 마라톤  - 매년 3월~11월 운영하며, 읽은 책 쪽수를 앱에 기록합니다. - 10,000쪽 달성 시 완주 인 ...
[2] (loan / 문의: lib_docs\02_loan.md)
    ## 제6조 상호대차  - 우리 도서관에 없는 책은 시내 다른 공공도서관에서 빌려 올 수 있습니다(상호대차). - 1인 월 5 ...
[3] (faq / 문의: lib_docs\01_faq.md)
    ### Q. 책을 잃어버렸어요. A. 같은 책(동일 판본)으로 변상하거나, 구입이 어려우면 정가의 1.5배를 현금으로 변상합니 ...


### 문제 C-4 — 애매한 질문은 필터로

"**예약**은 언제부터 돼요?" — 도서관에는 예약이 3종류입니다. 책 예약(loan), 스터디룸 예약(facility), 프로그램 신청(program).
이용자가 스터디룸 화면에서 물었다면 `facility`로 좁혀야 정확합니다.

In [17]:
q = "예약은 언제부터 돼요?"

print("───[ 전체 검색 ]───")
for r in lib_db.similarity_search(q, k=3):
    print(f" - [{r.metadata['category']}]", r.page_content[:50].replace("\n", " "), "...")

print("\n───[ facility만 ]───")
# TODO: category가 facility인 청크만
for r in lib_db.similarity_search(q, k=3, filter={"category": "facility"}):
    print(f" - [{r.metadata['category']}]", r.page_content[:50].replace("\n", " "), "...")

───[ 전체 검색 ]───
 - [loan] ## 제4조 연체  - 반납 예정일을 넘기면 **연체 일수만큼 대출이 정지**됩니다.    ...
 - [loan] - DVD 등 영상자료는 1인 2점, 7일간 대출합니다. - 신착 도서(입고 1개월 이내) ...
 - [faq] ### Q. 희망도서를 신청하고 싶어요. A. 홈페이지 > 희망도서 신청에서 1인 월 3권 ...

───[ facility만 ]───
 - [facility] ## 제5조 이용 제한  - 열람실·자료실에서 통화, 큰 소리 대화, 좌석 맡아두기는 금지 ...
 - [facility] ## 제2조 스터디룸  | 구분 | 인원 | 이용 시간 | |------|------|-- ...
 - [facility] # 한빛시립도서관 시설 이용 규정  ※ 교육용 가상 문서입니다. 실제 도서관과 무관합니다. ...


### 문제 C-5 — 관련도 점수로 규정 밖 질문 걸러내기

두 질문의 점수를 보고 `THRESHOLD`를 직접 정해 보세요.

In [18]:
THRESHOLD = 0.3     # TODO: 0.3에서 시작해 점수를 보며 조정

def safe_search(query, k=3):
    pairs = lib_db.similarity_search_with_score(query, k=k)                 # TODO: (문서, 관련도 점수)를 돌려주는 메서드
    kept = [(d, s) for d, s in pairs if s >= THRESHOLD]       # TODO: 점수 s가 THRESHOLD 이상
    print(f"질문: {query}   점수: {[round(s, 3) for _, s in pairs]}")
    if not kept:
        print("   → '규정에서 확인되지 않습니다. 안내데스크(031-000-1000)에 문의해 주세요.'\n")
    for d, s in kept:
        print(f"   → {s:.3f} [{d.metadata['category']}] {d.page_content[:40]!r}")
    print()
    return [d for d, _ in kept]

safe_search("스터디룸 몇 명까지 쓸 수 있어요?")
safe_search("도서관 근처 맛집 추천해 줘")

질문: 스터디룸 몇 명까지 쓸 수 있어요?   점수: [0.465, 0.654, 0.71]
   → 0.465 [facility] '## 제2조 스터디룸\n\n| 구분 | 인원 | 이용 시간 |\n|------'
   → 0.654 [facility] '# 한빛시립도서관 시설 이용 규정\n\n※ 교육용 가상 문서입니다. 실제 도'
   → 0.710 [facility] '## 제3조 노트북 존·멀티미디어실\n\n- 노트북 존(3층)은 전원 콘센트'

질문: 도서관 근처 맛집 추천해 줘   점수: [0.712, 0.724, 0.738]
   → 0.712 [facility] '# 한빛시립도서관 시설 이용 규정\n\n※ 교육용 가상 문서입니다. 실제 도'
   → 0.724 [loan] '## 제3조 반납\n\n- 운영시간에는 자료실 반납대에, 운영시간 외에는 정'
   → 0.738 [faq] '# 한빛시립도서관 자주 묻는 질문 (FAQ)\n\n※ 교육용 가상 문서입니다'



[Document(id='chunk-8', metadata={'team': '시설담당팀', 'category': 'facility', 'source': 'lib_docs\\03_facility.md'}, page_content='# 한빛시립도서관 시설 이용 규정\n\n※ 교육용 가상 문서입니다. 실제 도서관과 무관합니다.\n\n담당: 시설운영팀\n\n## 제1조 열람실\n\n- 운영시간: 화~일 07:00~23:00 (월요일 휴실)\n- 좌석은 키오스크나 모바일 앱으로 배정받으며, 1회 4시간 이용 후 연장할 수 있습니다.\n- 자리를 1시간 이상 비우면 좌석이 자동으로 반납됩니다.\n- 시험 기간(중간·기말고사 2주 전)에는 1인 1일 8시간까지로 제한합니다.\n\n## 제2조 스터디룸'),
 Document(id='chunk-5', metadata={'team': '자료봉사팀', 'category': 'loan', 'source': 'lib_docs\\02_loan.md'}, page_content='## 제3조 반납\n\n- 운영시간에는 자료실 반납대에, 운영시간 외에는 정문 옆 **무인반납함**에 반납합니다.\n- DVD 등 영상자료는 파손 위험이 있어 무인반납함에 넣을 수 없습니다.\n- **타관 반납**: 시내 다른 공공도서관 5곳에서도 반납할 수 있으며, 처리까지 2~3일이 걸립니다.\n\n## 제4조 연체'),
 Document(id='chunk-0', metadata={'team': '안내데스크', 'category': 'faq', 'source': 'lib_docs\\01_faq.md'}, page_content='# 한빛시립도서관 자주 묻는 질문 (FAQ)\n\n※ 교육용 가상 문서입니다. 실제 도서관과 무관합니다.\n\n### Q. 도서관 운영시간은 어떻게 되나요?\nA. 자료실은 화~금 09:00~20:00, 토·일 09:00~17:00 운영합니다.\n매주 **월요일과 법정 공휴일은 휴관**입니다.\n\n### Q. 회원증 없이 책을 빌

### 문제 C-6 — 이용자 말투 질문 10개로 정확도 측정

1등으로 찾은 청크의 분류가 정답과 같은지 자동 채점합니다. 마지막 2개는 직접 만들어 보세요.

In [19]:
test_questions = [
    ("책 늦게 돌려주면 벌금 내요?", "loan"),
    ("빌린 책 기간 더 늘릴 수 있어요?", "loan"),
    ("밤에 문 닫았는데 책 어디에 넣어요?", "loan"),
    ("공부방 몇 시까지 열어요?", "facility"),
    ("친구들이랑 모여서 공부할 방 있어요?", "facility"),
    ("차 몇 시간 공짜예요?", "facility"),
    ("다른 동네 살아도 가입돼요?", "membership"),
    ("오래 안 가면 회원 없어져요?", "membership"),
    ("도서관에서 하는 프로그램이 있어요?", "program"),     # TODO: 나만의 질문
    ("도서관에서 책을 빌릴 수 있나요?", "loan"),        # TODO: 질문과 정답 분류 모두 직접
]

correct = 0
for q, answer in test_questions:
    got = lib_db.similarity_search(q, k=1)[0].metadata["category"]     # TODO: 1등 하나만
    ok = (got == answer)
    correct += ok
    print(f"{'✅' if ok else '❌'} {q:<26} 정답={answer:<10} 검색={got}")

print(f"\n정확도: {correct}/{len(test_questions)} ({correct / len(test_questions):.0%})")

❌ 책 늦게 돌려주면 벌금 내요?           정답=loan       검색=program
❌ 빌린 책 기간 더 늘릴 수 있어요?        정답=loan       검색=faq
❌ 밤에 문 닫았는데 책 어디에 넣어요?       정답=loan       검색=program
✅ 공부방 몇 시까지 열어요?             정답=facility   검색=facility
❌ 친구들이랑 모여서 공부할 방 있어요?       정답=facility   검색=program
❌ 차 몇 시간 공짜예요?               정답=facility   검색=loan
✅ 다른 동네 살아도 가입돼요?            정답=membership 검색=membership
✅ 오래 안 가면 회원 없어져요?           정답=membership 검색=membership
✅ 도서관에서 하는 프로그램이 있어요?        정답=program    검색=program
✅ 도서관에서 책을 빌릴 수 있나요?         정답=loan       검색=loan

정확도: 5/10 (50%)


> ✍️ 틀린 질문이 있다면 이유는? (줄임말·구어체? FAQ와 겹침? 임베딩 모델 한계?)

### 문제 C-7 — MMR로 다양한 근거 가져오기

"처음 도서관 가는데 알아야 할 것 알려줘"는 회원증·대출·시설 등 **여러 측면**을 묻는 질문입니다.

In [20]:
q = "처음 도서관 가는데 알아야 할 것 알려줘"

sim = lib_db.as_retriever(search_kwargs={"k": 4})
mmr = lib_db.as_retriever(search_type="mmr", search_kwargs={"k": 4, "fetch_k": 20, "lambda_mult": 0.5})   # TODO: "mmr", 후보 20개

print("일반 유사도:", [d.metadata["category"] for d in sim.invoke(q)])
print("MMR       :", [d.metadata["category"] for d in mmr.invoke(q)])

일반 유사도: ['loan', 'faq', 'program', 'facility']
MMR       : ['loan', 'membership', 'program', 'program']


> 💡 MMR 쪽이 더 여러 분류를 가져오나요? `lambda_mult`를 1.0, 0.2로 바꿔 차이를 보세요.

---
# 🏁 파트 D — 종합: 검색 + 답변 = 완성된 책봇

지금까지 만든 것을 모두 연결합니다.

```
질문 → retriever(의미 검색) → 청크 3개 → format_docs(출처 붙이기) → rag_prompt → llm → 답변 + 출처
```

### 문제 D-1 — 검색한 청크에 출처를 붙여 한 덩어리로 만들기

In [22]:
def format_docs(docs):
    return "\n\n".join(
        # TODO: 파일 이름과 담당팀 꼬리표를 꺼내 출처 줄 만들기
        f"[출처: {os.path.basename(d.metadata['category'])} / 문의: {d.metadata['team']}]\n{d.page_content}"
        for d in docs
    )

retriever = lib_db.as_retriever(search_kwargs={"k": 3})
print(format_docs(retriever.invoke("스터디룸 몇 명까지 쓸 수 있어요?")))

[출처: facility / 문의: 시설담당팀]
## 제2조 스터디룸

| 구분 | 인원 | 이용 시간 |
|------|------|-----------|
| 소형 스터디룸 | 2~4인 | 1회 2시간 |
| 대형 스터디룸 | 5~8인 | 1회 3시간 |

- 홈페이지에서 **7일 전부터 전날까지** 예약할 수 있습니다. 당일 예약은 빈 방이 있을 때 안내데스크에서만 가능합니다.
- 1인 주 2회까지 예약할 수 있습니다.
- 예약 시간 15분이 지나도 오지 않으면 예약이 취소되고, 2회 이상이면 1개월간 예약할 수 없습니다.

## 제3조 노트북 존·멀티미디어실

[출처: facility / 문의: 시설담당팀]
# 한빛시립도서관 시설 이용 규정

※ 교육용 가상 문서입니다. 실제 도서관과 무관합니다.

담당: 시설운영팀

## 제1조 열람실

- 운영시간: 화~일 07:00~23:00 (월요일 휴실)
- 좌석은 키오스크나 모바일 앱으로 배정받으며, 1회 4시간 이용 후 연장할 수 있습니다.
- 자리를 1시간 이상 비우면 좌석이 자동으로 반납됩니다.
- 시험 기간(중간·기말고사 2주 전)에는 1인 1일 8시간까지로 제한합니다.

## 제2조 스터디룸

[출처: facility / 문의: 시설담당팀]
## 제3조 노트북 존·멀티미디어실

- 노트북 존(3층)은 전원 콘센트가 있는 좌석 40석이며, 자유석입니다.
- 멀티미디어실 PC는 1회 2시간, 1일 2회까지 사용할 수 있습니다.
- 출력은 흑백 1장 100원, 컬러 1장 300원이며 교통카드로 결제합니다.

## 제4조 주차

- 주차장은 60면이며, 도서관 이용자는 **2시간 무료**입니다.
- 2시간 이후 30분마다 500원이 부과됩니다.
- 무료 주차를 받으려면 출차 전 안내데스크에서 회원증을 확인받아야 합니다.
- 장애인·임산부·다자녀 가정은 4시간 무료입니다.


### 문제 D-2 — 책봇 완성: 출처 붙은 답변

💡 `{"document": retriever | format_docs, "question": RunnablePassthrough()}` 형태로 입력을 만듭니다.

In [23]:
from langchain_core.runnables import RunnablePassthrough

final_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "당신은 한빛시립도서관 안내 챗봇 '책봇'입니다. 아래 [자료]만 보고 친절한 존댓말로 답하세요.\n"
     "자료에 없으면 '규정에서 확인되지 않습니다. 안내데스크(031-000-1000)에 문의해 주세요.'라고 답하세요.\n"
     "답변 마지막 줄에 '(출처: 파일명 / 문의: 담당팀)'을 적으세요.\n\n"
     "[자료]\n{document}"),
    ("human", "{question}"),
])

book_bot = (
    {"document": retriever | format_docs, "question": RunnablePassthrough()}     # TODO: retriever와 format_docs 연결
    | final_prompt | llm | StrOutputParser()
)

for q in ["책 늦게 돌려주면 벌금 있어요?",
          "다른 동네 사는데 회원 가입 돼요?",
          "도서관 근처 맛집 추천해 줘"]:
    print("🙋", q)
    print("📚", book_bot.invoke(q))        # TODO: 실행 메서드
    print("-" * 60)

🙋 책 늦게 돌려주면 벌금 있어요?
📚 규정에서 확인되지 않습니다. 안내데스크(031-000-1000)에 문의해 주세요. (출처: faq / 문의: 안내데스크)
------------------------------------------------------------
🙋 다른 동네 사는데 회원 가입 돼요?
📚 한빛시에 거주하지 않으시는 경우, '준회원'으로 가입하실 수 있습니다. 준회원은 대출이 1인 3권까지 가능합니다. 더 궁금한 점이 있으시면 안내데스크(031-000-1000)에 문의해 주세요. 
(출처: membership / 문의: 자료봉사팀)
------------------------------------------------------------
🙋 도서관 근처 맛집 추천해 줘
📚 죄송하지만, 도서관 근처 맛집에 대한 정보는 규정에서 확인되지 않습니다. 안내데스크(031-000-1000)에 문의해 주세요. 
(출처: 파일명 / 문의: 담당팀)
------------------------------------------------------------


> 🎉 책봇 완성! 파트 A-1과 비교해 보세요. 같은 질문인데 이제는 **정확한 답 + 출처 + 문의처**가 나옵니다.

### (도전) 문제 D-3 — 나만의 개선

아래 중 하나를 골라 직접 고쳐 보세요.
1. `retriever`를 MMR로 바꾸기 (C-7)
2. 관련도 점수 기준을 넘는 청크만 넣기 (C-5의 `safe_search` 활용)
3. 어린이 회원 질문이면 쉬운 말로 답하도록 프롬프트 바꾸기

In [26]:
# 여기에 자유롭게 작성해 보세요

mmr_retriever = lib_db.as_retriever(search_type="mmr", search_kwargs={"k": 3})

THRESHOLD = 0.3

def safe_search(query, k=3):
    pairs = lib_db.similarity_search_with_score(query, k=k)                 # TODO: (문서, 관련도 점수)를 돌려주는 메서드
    kept = [(d, s) for d, s in pairs if s >= THRESHOLD]       # TODO: 점수 s가 THRESHOLD 이상
    print(f"질문: {query}   점수: {[round(s, 3) for _, s in pairs]}")
    if not kept:
        print("   → '규정에서 확인되지 않습니다. 안내데스크(031-000-1000)에 문의해 주세요.'\n")
    for d, s in kept:
        print(f"   → {s:.3f} [{d.metadata['category']}] {d.page_content[:40]!r}")
    print()
    return [d for d, _ in kept]

child_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "당신은 한빛시립도서관 안내 챗봇 '책봇'입니다. 아래 [자료]만 보고 친절한 존댓말로 답하세요.\n"
     "어린이 회원 질문이면 어린이 눈높이에 맞춰 답하세요.\n"
     "자료에 없으면 '규정에서 확인되지 않습니다. 안내데스크(031-000-1000)에 문의해 주세요.'라고 답하세요.\n"
     "답변 마지막 줄에 '(출처: 파일명 / 문의: 담당팀)'을 적으세요.\n\n"
     "[자료]\n{document}"),
    ("human", "{question}"),
])

child_book_bot = (
    {"document": mmr_retriever | format_docs, "question": RunnablePassthrough()}     # TODO: retriever와 format_docs 연결
    | child_prompt | llm | StrOutputParser()
)


for q in ["책 늦게 돌려주면 벌금 있어요?",
          "스터디룸 몇 명까지 쓸 수 있어요?",
          "어린이 회원은 몇 권까지 빌릴 수 있어요?",
          "어린이 독서교실은 몇 학년이 참여할 수 있어요?",
          "도서관 근처 맛집 추천해 줘"]:
    print("🙋", q)
    print("📚", child_book_bot.invoke(q))        # TODO: 실행 메서드
    print("-" * 60)

🙋 책 늦게 돌려주면 벌금 있어요?
📚 규정에서 확인되지 않습니다. 안내데스크(031-000-1000)에 문의해 주세요. (출처: faq / 문의: 안내데스크)
------------------------------------------------------------
🙋 스터디룸 몇 명까지 쓸 수 있어요?
📚 소형 스터디룸은 2명에서 4명까지 이용할 수 있고, 대형 스터디룸은 5명에서 8명까지 이용할 수 있습니다. 필요한 인원에 맞는 스터디룸을 예약하시면 됩니다. (출처: facility / 문의: 시설담당팀)
------------------------------------------------------------
🙋 어린이 회원은 몇 권까지 빌릴 수 있어요?
📚 규정에서 확인되지 않습니다. 안내데스크(031-000-1000)에 문의해 주세요. (출처: 파일명 / 문의: 담당팀)
------------------------------------------------------------
🙋 어린이 독서교실은 몇 학년이 참여할 수 있어요?
📚 어린이 독서교실은 초등학교 3학년부터 6학년까지의 친구들이 참여할 수 있어요. 방학 중에 2주 과정으로 진행된답니다! (출처: program / 문의: 프로그램팀)
------------------------------------------------------------
🙋 도서관 근처 맛집 추천해 줘
📚 죄송하지만, 도서관 근처 맛집에 대한 정보는 규정에서 확인되지 않습니다. 안내데스크(031-000-1000)에 문의해 주세요. 
(출처: 파일명 / 문의: 담당팀)
------------------------------------------------------------


---
# 🎓  정리

| 파트 | 핵심 코드 | 기억할 것 |
|------|-----------|-----------|
| A | `prompt \| llm \| StrOutputParser()`, `{document}` | 문서를 같이 주면 환각이 사라진다 |
| B | `DirectoryLoader` → `split_documents` → `metadata` | 쪼개고 꼬리표를 달아야 골라 쓸 수 있다 |
| C | `Chroma.from_documents`, `similarity_search`, `filter`, `..._with_relevance_scores`, MMR | 말투가 달라도 뜻으로 찾는다, 점수로 거른다 |
| D | `{"document": retriever \| format_docs, ...}` | 검색 + 출처 + 답변 = 실무형 RAG 챗봇 |

## 🤔 생각해보기

1. 도서관 운영시간이 바뀌면 책봇에서 무엇을 다시 해야 할까요? (파일 수정 → ? → ?)
2. C-4처럼 같은 단어(예약)가 여러 뜻으로 쓰일 때, 필터 말고 어떤 방법이 있을까요?
3. 책봇이 틀린 답을 했을 때 이용자가 입는 피해와, 현경봇(사내규정)이 틀렸을 때의 피해를 비교해 보세요.

> ✍️ 메모:
> - 1번: 문서 재분할(청킹) -> 임베딩 재생성 및 벡터 DB 갱신
> - 2번: 유저에게 되물어서 명확한 질문을 유도하기
> - 3번: 현경봇의 보안 관련 오답은 개인뿐 아니라 조직 전체에 영향을 줄 수 있다.